# 03 — Baseline Models
Popularity baseline + Matrix Factorization (SVD for explicit, ALS for implicit).
**Requires:** `02_data_preprocessing.ipynb` to have been run first.

In [ ]:
import pandas as pd
import numpy as np
import pickle, itertools, time
from pathlib import Path
from tqdm.notebook import tqdm

PROC = Path('data/processed')

def load_dataset(name):
    p = PROC / name
    train = pd.read_parquet(p / 'train.parquet')
    val   = pd.read_parquet(p / 'val.parquet')
    test  = pd.read_parquet(p / 'test.parquet')
    meta  = pickle.load(open(p / 'meta.pkl', 'rb'))
    print(f'Loaded {name}: n_users={meta["n_users"]:,}  n_items={meta["n_items"]:,}')
    return train, val, test, meta

ml_train,  ml_val,  ml_test,  ml_meta  = load_dataset('movielens_1m')
amz_train, amz_val, amz_test, amz_meta = load_dataset('amazon_music')

n_users_ml,  n_items_ml  = ml_meta['n_users'],  ml_meta['n_items']
n_users_amz, n_items_amz = amz_meta['n_users'], amz_meta['n_items']
all_items_ml = np.arange(n_items_ml)

## Evaluation metrics

In [ ]:
def hit_rate_at_k(recs, gt, k=10):
    hits, total = 0, 0
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        hits += int(len(set(r[:k]) & g) > 0)
        total += 1
    return hits / total if total else 0.0

def ndcg_at_k(recs, gt, k=10):
    scores = []
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        dcg  = sum(1/np.log2(i+2) for i,it in enumerate(r[:k]) if it in g)
        idcg = sum(1/np.log2(i+2) for i in range(min(len(g), k)))
        scores.append(dcg/idcg if idcg else 0.0)
    return float(np.mean(scores)) if scores else 0.0

def map_at_k(recs, gt, k=10):
    aps = []
    for u, r in recs.items():
        g = set(gt.get(u, []))
        if not g: continue
        hits, ps = 0, 0.0
        for rank, it in enumerate(r[:k]):
            if it in g:
                hits += 1; ps += hits/(rank+1)
        aps.append(ps / min(len(g), k))
    return float(np.mean(aps)) if aps else 0.0

def metrics(recs, gt, k=10):
    return {f'hr@{k}': hit_rate_at_k(recs,gt,k),
            f'ndcg@{k}': ndcg_at_k(recs,gt,k),
            f'map@{k}':  map_at_k(recs,gt,k)}

def cold_start_metrics(recs, test_df, k=10):
    gt_all = test_df.groupby('user_id')['item_id'].apply(list).to_dict()
    cold    = set(test_df[test_df.cold_start==True].user_id)
    regular = set(test_df[test_df.cold_start==False].user_id)
    return {
        'cold_start': metrics({u:r for u,r in recs.items() if u in cold},
                              {u:g for u,g in gt_all.items() if u in cold}, k),
        'regular':    metrics({u:r for u,r in recs.items() if u in regular},
                              {u:g for u,g in gt_all.items() if u in regular}, k),
    }

print('Evaluation functions defined.')

## Popularity baseline

In [ ]:
class PopularityBaseline:
    """Recommend globally most popular unseen items to every user."""
    def fit(self, train_df, item_col='item_id'):
        self.popular_items = train_df[item_col].value_counts().index.to_numpy()
        return self
    def recommend(self, user_ids, train_df, k=10, user_col='user_id', item_col='item_id'):
        seen = train_df.groupby(user_col)[item_col].apply(set).to_dict()
        return {u: [it for it in self.popular_items if it not in seen.get(u,set())][:k]
                for u in user_ids}

# MovieLens
pop_ml = PopularityBaseline().fit(ml_train)
pop_recs_ml = pop_ml.recommend(ml_test.user_id.unique(), ml_train, k=10)
gt_ml = ml_test.groupby('user_id')['item_id'].apply(list).to_dict()
pop_m_ml = metrics(pop_recs_ml, gt_ml)
print('Popularity — MovieLens 1M:', pop_m_ml)

# Amazon Music
pop_amz = PopularityBaseline().fit(amz_train)
pop_recs_amz = pop_amz.recommend(amz_test.user_id.unique(), amz_train, k=10)
gt_amz = amz_test.groupby('user_id')['item_id'].apply(list).to_dict()
pop_m_amz = metrics(pop_recs_amz, gt_amz)
print('Popularity — Amazon Music: ', pop_m_amz)

## SVD — explicit feedback (MovieLens)

In [ ]:
from surprise import Dataset, Reader, SVD as SurpriseSVD

def fit_svd(train_df, n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02):
    reader   = Reader(rating_scale=(0,1))
    surprise_df = train_df[['user_id','item_id','label']].rename(
        columns={'user_id':'userID','item_id':'itemID','label':'rating'})
    data     = Dataset.load_from_df(surprise_df, reader)
    trainset = data.build_full_trainset()
    model    = SurpriseSVD(n_factors=n_factors, n_epochs=n_epochs,
                            lr_all=lr_all, reg_all=reg_all, verbose=False)
    t0 = time.time()
    model.fit(trainset)
    print(f'  SVD fitted {time.time()-t0:.1f}s | factors={n_factors} reg={reg_all}')
    return model

def svd_recommend(model, user_ids, train_df, all_items, k=10):
    seen = train_df.groupby('user_id')['item_id'].apply(set).to_dict()
    recs = {}
    for uid in user_ids:
        candidates = [it for it in all_items if it not in seen.get(uid,set())]
        scores     = np.array([model.predict(str(uid),str(it)).est for it in candidates])
        top_k      = np.argsort(scores)[::-1][:k]
        recs[uid]  = [candidates[i] for i in top_k]
    return recs

print('SVD functions defined.')

In [ ]:
# Tune SVD on validation set
val_users_ml = ml_val.user_id.unique()
gt_val_ml    = ml_val.groupby('user_id')['item_id'].apply(list).to_dict()
svd_results  = []

for n_factors, reg_all in tqdm(list(itertools.product([50,100,200],[0.01,0.02,0.05])),
                                desc='SVD tuning'):
    m     = fit_svd(ml_train, n_factors=n_factors, reg_all=reg_all)
    recs  = svd_recommend(m, val_users_ml, ml_train, all_items_ml, k=10)
    svd_results.append({'n_factors':n_factors,'reg_all':reg_all,
                        **metrics(recs, gt_val_ml)})

svd_df = pd.DataFrame(svd_results).sort_values('ndcg@10', ascending=False)
print('SVD tuning results:')
svd_df

In [ ]:
# Best SVD on test set
best = svd_df.iloc[0]
best_svd = fit_svd(ml_train, n_factors=int(best.n_factors), reg_all=float(best.reg_all))
svd_recs_ml = svd_recommend(best_svd, ml_test.user_id.unique(), ml_train, all_items_ml, k=10)
svd_m_ml    = metrics(svd_recs_ml, gt_ml)
print(f'Best SVD (factors={int(best.n_factors)}, reg={best.reg_all}) test metrics:')
print(svd_m_ml)

print('\nCold-start breakdown:')
for group, m in cold_start_metrics(svd_recs_ml, ml_test).items():
    print(f'  {group}: {m}')

## ALS — implicit feedback (Amazon Music)

In [ ]:
import implicit
import scipy.sparse as sp

def build_sparse(train_df, n_users, n_items):
    counts = train_df.groupby(['user_id','item_id']).size().reset_index(name='c')
    return sp.csr_matrix((counts.c.values,
                          (counts.user_id.values, counts.item_id.values)),
                         shape=(n_users, n_items))

def fit_als(train_df, n_users, n_items,
            factors=128, iterations=20, regularization=0.01, alpha=40.0):
    user_item = build_sparse(train_df, n_users, n_items)
    model = implicit.als.AlternatingLeastSquares(
        factors=factors, iterations=iterations,
        regularization=regularization, alpha=alpha, use_gpu=False)
    t0 = time.time()
    model.fit(user_item.T.tocsr())
    print(f'  ALS fitted {time.time()-t0:.1f}s | factors={factors} alpha={alpha}')
    return model, user_item

def als_recommend(model, user_item, user_ids, k=10):
    recs = {}
    for uid in user_ids:
        ids, _ = model.recommend(uid, user_item[uid], N=k,
                                 filter_already_liked_items=True)
        recs[uid] = ids.tolist()
    return recs

print('ALS functions defined.')

In [ ]:
# Tune ALS on validation set
val_users_amz = amz_val.user_id.unique()
gt_val_amz    = amz_val.groupby('user_id')['item_id'].apply(list).to_dict()
als_results   = []

for factors, alpha in tqdm(list(itertools.product([64,128,256],[1.0,10.0,40.0])),
                            desc='ALS tuning'):
    model, ui = fit_als(amz_train, n_users_amz, n_items_amz,
                        factors=factors, alpha=alpha)
    recs = als_recommend(model, ui, val_users_amz, k=10)
    als_results.append({'factors':factors,'alpha':alpha,
                        **metrics(recs, gt_val_amz)})

als_df = pd.DataFrame(als_results).sort_values('ndcg@10', ascending=False)
print('ALS tuning results:')
als_df

In [ ]:
# Best ALS on test set
best_als_row = als_df.iloc[0]
best_als_model, best_ui = fit_als(
    amz_train, n_users_amz, n_items_amz,
    factors=int(best_als_row.factors), alpha=float(best_als_row.alpha))
als_recs_amz = als_recommend(best_als_model, best_ui, amz_test.user_id.unique(), k=10)
als_m_amz    = metrics(als_recs_amz, gt_amz)
print(f'Best ALS (factors={int(best_als_row.factors)}, alpha={best_als_row.alpha}) test metrics:')
print(als_m_amz)

print('\nCold-start breakdown:')
for group, m in cold_start_metrics(als_recs_amz, amz_test).items():
    print(f'  {group}: {m}')

## Results summary

In [ ]:
summary = pd.DataFrame([
    {'Dataset':'MovieLens 1M',  'Model':'Popularity', **pop_m_ml},
    {'Dataset':'MovieLens 1M',  'Model':'SVD (best)', **svd_m_ml},
    {'Dataset':'Amazon Music',  'Model':'Popularity', **pop_m_amz},
    {'Dataset':'Amazon Music',  'Model':'ALS (best)', **als_m_amz},
]).round(4)

summary.to_csv(PROC / 'baseline_results.csv', index=False)
print('Results saved to data/processed/baseline_results.csv')
summary

In [ ]:
# Save best model configs for use in deep learning notebooks
baseline_config = {
    'best_svd': {'n_factors': int(svd_df.iloc[0].n_factors),
                 'reg_all':   float(svd_df.iloc[0].reg_all)},
    'best_als': {'factors': int(als_df.iloc[0].factors),
                 'alpha':   float(als_df.iloc[0].alpha)},
    'baseline_metrics': {
        'movielens_popularity': pop_m_ml,
        'movielens_svd':        svd_m_ml,
        'amazon_popularity':    pop_m_amz,
        'amazon_als':           als_m_amz,
    }
}
with open(PROC / 'baseline_config.pkl', 'wb') as f:
    pickle.dump(baseline_config, f)
print('Baseline config saved. Run 04_ncf.ipynb next.')